# EXP-01 — Colab CPU bootstrap
Use a new CPU runtime. Upload exp01-colab-handoff.zip through Files, or mount Drive yourself after reviewing its broad permissions. Enter the trusted SHA from the local handoff receipt. This runs the SAME models in an isolated venv; it does not install into the host kernel. Pilot only, not official research.
After Run all, download the verified output ZIP or copy the exact output allowlist to your personal Drive folder. Do not share input/cache/credentials or mark local execution as Colab evidence.

In [ ]:
import hashlib, json, os, re, stat, subprocess, sys, uuid, zipfile
from pathlib import Path, PurePosixPath
import google.colab
assert (3, 11) <= sys.version_info[:2] <= (3, 13), 'Unsupported Python: stop; do not remove pins'
ARCHIVE = Path(input('Absolute path to handoff ZIP in Colab: ').strip())
EXPECTED_SHA = input('Trusted ZIP SHA256 from local receipt: ').strip()
RECIPIENT = input('HoaiTam / rosy179 / ThanhTris: ').strip()
assert RECIPIENT in {'HoaiTam', 'rosy179', 'ThanhTris'}
assert re.fullmatch('[a-f0-9]{64}', EXPECTED_SHA) and hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() == EXPECTED_SHA, 'ZIP pin mismatch'
SESSION = uuid.uuid4().hex
WORK = Path('/content/ie212-exp01') / SESSION
INPUT = WORK / 'input'
with zipfile.ZipFile(ARCHIVE) as z:
    members = z.infolist()
    assert 0 < len(members) <= 128 and len(members) == len({m.filename for m in members})
    assert sum(m.file_size for m in members) <= 8*1024**2
    for m in members:
        p = PurePosixPath(m.filename)
        assert not p.is_absolute() and '..' not in p.parts and '\\' not in m.filename and not re.match('^[A-Za-z]:', m.filename)
        assert not stat.S_ISLNK(m.external_attr >> 16) and not m.flag_bits & 1
    INPUT.mkdir(parents=True, exist_ok=False)
    z.extractall(INPUT)
manifest = json.loads((INPUT / 'handoff.json').read_text())
for name, pin in manifest['files'].items():
    assert hashlib.sha256((INPUT / name).read_bytes()).hexdigest() == pin
assert {p.relative_to(INPUT).as_posix() for p in INPUT.rglob('*') if p.is_file()} == set(manifest['files']) | {'handoff.json'}
CODE = INPUT / 'code'
print('PINNED_HANDOFF_BYTES_VERIFIED', SESSION)


In [ ]:
ENV = WORK / 'venv'
def checked(args):
    completed = subprocess.run(args, capture_output=True, text=True)
    print(completed.stdout)
    if completed.returncode: print(completed.stderr)
    completed.check_returncode()
checked([sys.executable, '-m', 'venv', '--without-pip', str(ENV)])
PYTHON = ENV / 'bin/python'
checked([sys.executable, '-m', 'pip', '--isolated', '--python', str(PYTHON), 'install', '--disable-pip-version-check', '-r', str(CODE / 'notebooks/requirements-colab-lock.txt')])
checked([str(PYTHON), '-m', 'pip', 'check'])
print('ISOLATED_CPU_ENV_READY')


In [ ]:
OUTPUT = WORK / 'output'
checked([str(PYTHON), '-B', str(CODE / 'scripts/exp01-colab.py'), 'run', '--handoff-root', str(INPUT), '--output', str(OUTPUT), '--recipient', RECIPIENT])
report = json.loads((OUTPUT / 'colab-report.json').read_text())
assert report['status'] == 'COLAB_HARNESS_PASSED' and report['runtime_kind'] == 'GOOGLE_COLAB'
assert report['research_ready'] is False and report['official_exported'] is False
print('COLAB_HARNESS_PASSED', {a: p['status'] for a, p in report['parity'].items()})


In [ ]:
pins = json.loads((OUTPUT / 'output-checksums.json').read_text())
OUTPUT_ZIP = WORK / 'exp01-colab-output.zip'
with zipfile.ZipFile(OUTPUT_ZIP, 'x', compression=zipfile.ZIP_DEFLATED) as z:
    for name in sorted(set(pins) | {'output-checksums.json'}):
        source = OUTPUT / name
        if name in pins: assert hashlib.sha256(source.read_bytes()).hexdigest() == pins[name]
        z.write(source, name)
with zipfile.ZipFile(OUTPUT_ZIP) as z:
    for name, pin in pins.items(): assert hashlib.sha256(z.read(name)).hexdigest() == pin
print('OUTPUT_ZIP_READBACK_VERIFIED', hashlib.sha256(OUTPUT_ZIP.read_bytes()).hexdigest())
from google.colab import files
files.download(str(OUTPUT_ZIP))
